In [1]:
import json

In [2]:
file1 = 'gpt-5.1/question_responses1.json'
with open(file1, 'r') as f:
    data1 = json.load(f)

len(data1)

200

In [3]:
file2 = 'gpt-5.1/question_responses2.json'
with open(file2, 'r') as f:
    data2 = json.load(f)

len(data2)

168

In [4]:
combined_data = data1 + data2
len(combined_data)

368

In [6]:
unique_ids = {}
for review in combined_data:
    review_id = review["ReviewID"]
    if review_id not in combined_data:
        unique_ids[review_id] = review

print(len(unique_ids))

368


In [5]:
with open("gpt-5.1/question_responses.json", "w", encoding='utf-8') as file:
    json.dump(combined_data, file, indent=4)

# For Analysis Testing

## SAMPLING DATA

In [1]:
import json

claude_file = "claude_4.5_sonnet/question_responses.json"
gpt_file = "gpt-5.1/question_responses.json"
qwen3_4B_file = "qwen3-4B/question_responses.json"
llama4_file = "api-llama4/question_responses.json"
huatuo_7B_file = "huatuo-7B/question_responses.json"

def load_file(file_name):
    with open(file_name, 'r') as f:
        data = json.load(f)
    return data

claude_data = load_file(claude_file)
gpt_data = load_file(gpt_file)
qwen3_4B_data = load_file(qwen3_4B_file)
llama4_data = load_file(llama4_file)
huatuo_7B_data = load_file(huatuo_7B_file)

In [2]:
def format_outputs(raw_data: list[dict]) -> dict:
    grouped = {}
    
    for item in raw_data:
        review_id = item["ReviewID"]
        q_and_a = item["ModelGeneratedAnswersWithQuestions"]

        for k, v in q_and_a.items():
            if k != "multiturn":
                pair_key = f"{review_id}_{k}"
                if pair_key not in grouped:
                    grouped[pair_key] = {}
                grouped[pair_key]["positive_answer"] = v["positive_answer"]
                grouped[pair_key]["negative_answer"] = v["negative_answer"]
            else: # multiturn situation
                pos_answers = v["positive_answer"]
                neg_answers = v["negative_answer"]
                pair_key = f"{review_id}_{k}"
                for index, item in enumerate(list(zip(pos_answers, neg_answers))):
                    current_pair_key = f"{pair_key}_{index}"
                    if pair_key not in grouped:
                        grouped[current_pair_key] = {}
                    grouped[current_pair_key]["positive_answer"] = item[0]
                    grouped[current_pair_key]["negative_answer"] = item[1]

    return grouped

In [3]:
claude_data = format_outputs(claude_data)
gpt_data = format_outputs(gpt_data)
qwen3_4B_data = format_outputs(qwen3_4B_data)
llama4_data = format_outputs(llama4_data)
huatuo_7B_data = format_outputs(huatuo_7B_data)


print(len(claude_data))
print(len(gpt_data))
print(len(qwen3_4B_data))
print(len(llama4_data))
print(len(huatuo_7B_data))

5888
5888
5888
5888
5888


In [4]:
import random

def get_ten_random(data):
    random_keys = random.sample(list(data.keys()), 2)
    subset = {k: data[k] for k in random_keys}
    return subset

claude_sample = get_ten_random(claude_data)
gpt_sample = get_ten_random(gpt_data)
qwen3_4B_sample = get_ten_random(qwen3_4B_data)
llama4_sample = get_ten_random(llama4_data)
huatuo_7B_sample = get_ten_random(huatuo_7B_data)

In [5]:
# Merge multiple dictionaries
combined = {**claude_sample, **gpt_sample, **qwen3_4B_sample, **llama4_sample, **huatuo_7B_sample}
print(len(combined))

10


In [6]:
with open("sample_for_hedging_pilot_annotations.json", "w", encoding='utf-8') as file:
    json.dump(combined, file, indent=4)

In [14]:
with open("sample_for_analysis_test.json", "w", encoding='utf-8') as file:
    json.dump(combined, file, indent=4)

## COMPARE HUMAN ANNOTATION and LLM